In [2]:
import pandas as pd 
df = pd.read_csv("../data/transactions.csv")
df.head()

,transaction_id,customer_id,amount,hour,day_of_week,distance_from_home,merchant_category,is_fraud
0,TXN_00000000,861,321.657107,12,6,8.349037,restaurant,0
1,TXN_00000001,3773,1944.835714,18,2,4.499823,grocery,0
2,TXN_00000002,3093,14551.485133,11,2,18.237703,grocery,0
3,TXN_00000003,467,977.794274,6,1,2.069479,restaurant,0
4,TXN_00000004,4427,106.712706,20,1,13.467285,restaurant,0


## Feature Selection

The same numerical features used for Isolation Forest are used here to
show comparison between the two models.

In [3]:
features = [
    "amount",
    "hour",
    "day_of_week",
    "distance_from_home"
]
X = df[features]

In [4]:
from sklearn.svm import OneClassSVM
ocs = OneClassSVM(
    kernel = "rbf",
    gamma = "scale",
    nu = 0.01
)
ocs.fit(X)

OneClassSVM(nu=0.01)

In [5]:
df["svm_label"]=ocs.predict(X)
df[["is_fraud","svm_label"]].value_counts()

is_fraud  svm_label
0          1           97775
1          1            1173
0         -1             968
1         -1              84
Name: count, dtype: int64

### Observation Before Scaling

The One-Class SVM model was trained without feature scaling as an experiment. Compared to Isolation Forest, training and prediction were
much slower.

One Class SVM measures the distance between the data points and it tries to learn from the boundary around the normal data.It keeps how far each transaction is from the normal pattern. So, when features are not scaled , the features with big numbers dominate everything and it makes wrong decisions. However, Isolation forest does not work on numerical value and hence scaling was not needed

When svm_label = 1 then it is predicted normal 
When svm_label = -1 then it is predicted anomaly 

correctly predicted normal transactions are : 97775---> actual users not disturbed 
fraud transactions but predicted as normal transactions : 1173---> false -ve
normal transactions but predicted as fraudulent transactions : 968--->false +ve
correctly predicted fraudulent transactions : 84---> correctly detected fraud 

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [8]:
ocs_scaled = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.01
)

ocs_scaled.fit(X_scaled)

OneClassSVM(nu=0.01)

In [10]:
df["svm_label_scaled"] = ocs_scaled.predict(X_scaled)
df[["is_fraud", "svm_label_scaled"]].value_counts()


is_fraud  svm_label_scaled
0          1                  97838
1          1                   1162
0         -1                    905
1         -1                     95
Name: count, dtype: int64

### Observation After Scaling

After applying StandardScaler, One-Class SVM showed improved stability during
training and prediction.
But still after scaling , One-Class SVM remains slower than Isolation Forest.
The feature selection has improved the performance of One Class SVM but there still exists other constraints. 

Scaling has visibly improved the behavior compared to before.However speed and operational cost are important for real time fraud detection, One Class SVM is not suitable for real time system 

In [11]:
df["svm_predicted_fraud"] = (df["svm_label_scaled"] == -1).astype(int)
df[["is_fraud", "svm_predicted_fraud"]].head()

,is_fraud,svm_predicted_fraud
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


here we converted svm labels to fraud prediction. Simple reason as to why I converted : 
In One Class SVM we had 1 means normal and -1 means anomaly
but our dataset has is_fraud = 1 or 0 where 1 means fraud and 0 means normal
Because Evaluation Metrics work on 0 / 1 , I did the conversion.

In [12]:
# how many frauds the model predicted
df["svm_predicted_fraud"].value_counts()

svm_predicted_fraud
0    99000
1     1000
Name: count, dtype: int64

99,000 are normal transactions
1,000 are flagged as anomalies

In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score

svm_precision = precision_score(df["is_fraud"], df["svm_predicted_fraud"])
svm_recall = recall_score(df["is_fraud"], df["svm_predicted_fraud"])
svm_f1 = f1_score(df["is_fraud"], df["svm_predicted_fraud"])

svm_precision, svm_recall, svm_f1

(0.095, 0.07557677008750995, 0.08418254319893664)

In [16]:
from sklearn.metrics import roc_auc_score
svm_scores = ocs_scaled.decision_function(X_scaled)
svm_auc = roc_auc_score(df["is_fraud"], -svm_scores)
svm_auc

0.6218764620685355

### Precision, Recall, F1, ROC-AUC

Precision says how many transactions flagged as fraud were actually
fraudulent. In here, low precision means more innocent people get blocked (false +ve)
Recall says how many actual fraud cases were detected.
F1-score balances precision and recall.
ROC-AUC tells how good the model is at separating the frauds from normal users. If AUC is higher then the model is better at understanding the difference. 

### Isolation forest VS One Class SVM
Isolation Forest is better because it makes less mistakes on normal
transactions. It also understands the fraud patterns and works faster.